# Unlimited-OCR 2 (A100)

Drive-native PDF/image batch OCR; official Transformers backend, local SSD model and kernels, persistent Drive results, Caddy auth and asynchronous CLI jobs.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "unlimited-ocr")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8000"))
AUTH         = os.environ.get("CT_AUTH", "caddy")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "18080"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
OCR_CACHE_SOURCE = '"""Stage optional compiler caches using one Drive archive per environment/GPU."""\nimport hashlib\nimport os\nimport shutil\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef _progress(log, message, operation):\n    stopped = threading.Event()\n    started = time.monotonic()\n\n    def heartbeat():\n        while not stopped.wait(10):\n            log(f\'{message}, elapsed {time.monotonic()-started:.0f} s\')\n\n    log(message)\n    thread = threading.Thread(target=heartbeat, daemon=True)\n    thread.start()\n    try:\n        return operation()\n    finally:\n        stopped.set()\n        thread.join()\n\n\ndef _files(root):\n    return sorted(p for p in root.rglob(\'*\') if p.is_file() and not p.is_symlink()\n                  and not p.name.endswith((\'.lock\', \'.tmp\')))\n\n\ndef _fingerprint(root, files):\n    digest = hashlib.sha256()\n    for path in files:\n        stat = path.stat()\n        digest.update(f\'{path.relative_to(root)}:{stat.st_size}:{stat.st_mtime_ns}\\n\'.encode())\n    return digest.hexdigest()\n\n\ndef _unpack(archive, local):\n    # Validate the whole archive before extracting; compiler caches contain\n    # regular files/directories only. Never follow archive or local symlinks.\n    with tarfile.open(archive, \'r:\') as tar:\n        members = tar.getmembers()\n        root = local.resolve()\n        for member in members:\n            destination = local / member.name\n            if (not (member.isfile() or member.isdir()) or Path(member.name).is_absolute()\n                    or not destination.resolve().is_relative_to(root)):\n                raise ValueError(\'Cache archive contains unsafe paths or file types\')\n            current = destination\n            while current != local:\n                if current.is_symlink():\n                    raise ValueError(\'Cache target contains a symbolic link\')\n                current = current.parent\n        tar.extractall(local, members=members, filter=\'data\')\n\n\ndef restore_cache(local, legacy, archive_dir, key, log=print):\n    local, legacy, archive_dir = map(Path, (local, legacy, archive_dir))\n    local.mkdir(parents=True, exist_ok=True)\n    archive = archive_dir / key / (local.name + \'.tar\')\n    stamp = local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')\n    if archive.is_file():\n        stat = archive.stat()\n        signature = f\'{stat.st_size}:{stat.st_mtime_ns}\'\n        if stamp.exists() and stamp.read_text() == signature and _files(local):\n            log(f\'{local.name}: already restored locally, skipping duplicate transfer\')\n            return \'local\'\n        try:\n            with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-restore-\') as tmp:\n                staged = Path(tmp) / \'cache.tar\'\n                _progress(log, f\'{local.name}: reading archive from Drive ({stat.st_size/1e6:.1f} MB)\',\n                          lambda: shutil.copyfile(archive, staged))\n                _progress(log, f\'{local.name}: unpacking on local SSD\', lambda: _unpack(staged, local))\n            stamp.write_text(signature)\n            (local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')).write_text(\n                _fingerprint(local, _files(local)))\n            log(f\'{local.name}: archive restore complete\')\n            return \'archive\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: archive unavailable, falling back to the legacy cache ({error})\')\n    if legacy.is_dir():\n        try:\n            def migrate():\n                shutil.copytree(legacy, local, dirs_exist_ok=True, symlinks=True)\n            _progress(log, f\'{local.name}: first-time migration of the legacy Drive cache (slow when there are many small files)\', migrate)\n            if _files(local):\n                save_cache(local, archive_dir, key, log)\n                return \'legacy\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: legacy cache migration failed, falling back to on-demand generation ({error})\')\n    log(f\'{local.name}: no persistent cache; the service will generate it on demand\')\n    return \'miss\'\n\n\ndef save_cache(local, archive_dir, key, log=print):\n    local, archive_dir = Path(local), Path(archive_dir)\n    files = _files(local)\n    if not files:\n        log(f\'{local.name}: no cache files, skipping archive\')\n        return False\n    fingerprint = _fingerprint(local, files)\n    saved = local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')\n    destination = archive_dir / key / (local.name + \'.tar\')\n    if saved.exists() and saved.read_text() == fingerprint and destination.is_file():\n        log(f\'{local.name}: content unchanged, skipping upload\')\n        return False\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    try:\n        with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-save-\') as tmp:\n            staged = Path(tmp) / \'cache.tar\'\n            def pack():\n                with tarfile.open(staged, \'w:\') as tar:\n                    for path in files:\n                        tar.add(path, arcname=str(path.relative_to(local)), recursive=False)\n            _progress(log, f\'{local.name}: packing {len(files)} files locally\', pack)\n            _progress(log, f\'{local.name}: saving archive to Drive ({staged.stat().st_size/1e6:.1f} MB)\',\n                      lambda: shutil.copyfile(staged, temporary))\n            os.replace(temporary, destination)\n        saved.write_text(fingerprint)\n        stat = destination.stat()\n        (local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')).write_text(\n            f\'{stat.st_size}:{stat.st_mtime_ns}\')\n        log(f\'{local.name}: Drive archive updated\')\n        return True\n    finally:\n        temporary.unlink(missing_ok=True)\n'
# @title 1. Setup -- pinned official Transformers backend; model on local SSD
from pathlib import Path

OCR_WORK = Path('/content/unlimited-ocr')
OCR_WORK.mkdir(exist_ok=True)
if not shutil.which('uv'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
OCR_PY = str(OCR_WORK / 'venv/bin/python')
if not Path(OCR_PY).exists():
    subprocess.run(['uv', 'venv', '--python', '3.10', str(OCR_WORK / 'venv')], check=True)
OCR_PACKAGES = ['torch==2.13.0', 'torchvision', 'pillow', 'transformers==4.52.1',
                'huggingface-hub==0.36.0', 'tokenizers==0.21.4', 'addict==2.4.0',
                'einops==0.8.2', 'easydict==1.13', 'pymupdf==1.27.2.2', 'requests']
OCR_MARKER = OCR_WORK / '.installed'
OCR_SIGNATURE = json.dumps(OCR_PACKAGES)
if not OCR_MARKER.exists() or OCR_MARKER.read_text() != OCR_SIGNATURE:
    subprocess.run(['uv', 'pip', 'install', '--python', OCR_PY, *OCR_PACKAGES], check=True)
    OCR_MARKER.write_text(OCR_SIGNATURE)
OCR_MODEL = OCR_WORK / 'model'
subprocess.run([OCR_PY, '-c', '''from huggingface_hub import snapshot_download
snapshot_download('baidu/Unlimited-OCR-2', revision='0480966938291445591f69f08a13b041c073a96a',
                  local_dir='/content/unlimited-ocr/model',
                  ignore_patterns=['assets/*', 'wheel/*', '*.pdf', '*.md'])
'''], check=True, env={**os.environ, 'HF_XET_CHUNK_CACHE_SIZE_BYTES': '0'})
shutil.rmtree('/root/.cache/huggingface/xet', ignore_errors=True)
OCR_POSTPROCESS = OCR_WORK / 'process_unlimitedOCR2.py'
if not OCR_POSTPROCESS.exists():
    with urllib.request.urlopen(
        'https://raw.githubusercontent.com/baidu/Unlimited-OCR/'
        'ea42e6d15ff93fdd0e8041fc0ac70880513453d3/utils/process_unlimitedOCR2.py',
        timeout=60,
    ) as _src, open(OCR_POSTPROCESS, 'wb') as _dst:
        shutil.copyfileobj(_src, _dst)
# Triton / Inductor compile on SSD; restore and persist caches by GPU and environment.
(OCR_WORK / 'colab_cache.py').write_text(OCR_CACHE_SOURCE)
sys.path.insert(0, str(OCR_WORK))
from colab_cache import restore_cache, save_cache
try:
    OCR_GPU = (
        subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
        .splitlines()[0]
        .strip()
        .replace(' ', '_')
    )
except (OSError, subprocess.SubprocessError, IndexError):
    OCR_GPU = 'cpu'
OCR_CACHE_KEY = 'torch213-uocr2-04809669-' + OCR_GPU
OCR_CACHE_PAIRS = []
for name in ('triton-cache', 'inductor-cache'):
    local = OCR_WORK / name
    try:
        restore_cache(local, Path(DATA_DIR) / name, Path(DATA_DIR) / 'kernel-archives', OCR_CACHE_KEY)
    except (OSError, ValueError) as _cache_err:
        print(f'Warning: cache restore skipped for {name}: {_cache_err}', flush=True)
    OCR_CACHE_PAIRS.append(local)
print('Unlimited-OCR 2 setup complete', flush=True)


In [ ]:
# @title 2. Unlimited-OCR Drive job service
(OCR_WORK / 'ocr_parameters.py').write_text('"""Public inference settings for the pinned Unlimited-OCR 2 Transformers model.\n\nOnly tokenizer, image_file(s), and output_path are owned by the job runner.\n"""\nimport argparse\nimport json\nimport math\n\n# name -> (JSON type, minimum, nullable). No upper caps beyond the model itself.\nPARAMETERS = {\n    \'prompt\': (str, None, False),\n    \'base_size\': (int, 1, False),\n    \'image_size\': (int, 1, False),\n    \'crop_mode\': (bool, None, True),\n    \'test_compress\': (bool, None, False),\n    \'save_results\': (bool, None, False),\n    \'eval_mode\': (bool, None, False),\n    \'max_length\': (int, 1, False),\n    \'max_new_tokens\': (int, 1, True),\n    \'tps_interval\': (int, 0, False),\n    \'no_repeat_ngram_size\': (int, 0, False),\n    \'ngram_window\': (int, 0, False),\n    \'ngram_whitelist_token_ids\': (list, None, True),\n    \'temperature\': (float, 0, False),\n    \'page_count_guard\': (bool, None, False),\n    \'max_page_tokens\': (int, None, False),\n    \'dash_topm\': (int, 1, False),\n    \'max_crops\': (int, 1, False),\n    \'return_dict\': (bool, None, False),\n}\n\n\ndef validate_parameters(value):\n    if not isinstance(value, dict):\n        raise ValueError(\'inference_params must be a JSON object\')\n    for name, item in value.items():\n        if name not in PARAMETERS:\n            raise ValueError(f\'Unsupported inference parameter: {name}; available: {", ".join(PARAMETERS)}\')\n        kind, minimum, nullable = PARAMETERS[name]\n        if item is None and nullable:\n            continue\n        valid = type(item) is kind\n        if kind is float:\n            valid = type(item) in (int, float) and math.isfinite(item)\n        if not valid or (minimum is not None and item < minimum):\n            raise ValueError(f\'{name}: type must be {kind.__name__}\' + (f\', value >= {minimum}\' if minimum is not None else \'\'))\n        if kind is list and any(type(n) is not int or n < 0 for n in item):\n            raise ValueError(f\'{name} must be an array of non-negative integers or null\')\n    return dict(value)\n\n\ndef inference_kwargs(params, page_count, tokens_per_page):\n    """Keep historical service defaults; explicit values, including null, always win."""\n    result = dict(eval_mode=True, crop_mode=False, return_dict=True,\n                  max_new_tokens=min(120000, tokens_per_page * page_count),\n                  no_repeat_ngram_size=35, ngram_window=1024 if page_count > 1 else 128)\n    result.update(validate_parameters(params))\n    return result\n\n\ndef add_inference_arguments(parser):\n    parser.add_argument(\'--params-file\', help=\'local JSON file with inference parameters (official parameter names)\')\n    parser.add_argument(\'--params-json\', help=\'JSON object with inference parameters; overrides the parameter file\')\n    parser.add_argument(\'--param\', action=\'append\', default=[], metavar=\'KEY=JSON\',\n                        help=\'repeatable; overrides the JSON configuration, e.g. --param crop_mode=null\')\n    for name, (kind, _, _) in PARAMETERS.items():\n        flag = \'--\' + name.replace(\'_\', \'-\')\n        if kind is bool:\n            parser.add_argument(flag, dest=name, action=argparse.BooleanOptionalAction, default=None)\n        elif kind is list:\n            parser.add_argument(flag, dest=name, type=json.loads, default=None, metavar=\'JSON_ARRAY\')\n        else:\n            parser.add_argument(flag, dest=name, type=kind, default=None)\n\n\ndef cli_parameters(args):\n    from pathlib import Path\n    result = {}\n    for raw in ([Path(args.params_file).expanduser().read_text(encoding=\'utf-8\')] if args.params_file else []) + ([args.params_json] if args.params_json else []):\n        result.update(validate_parameters(json.loads(raw)))\n    for name in PARAMETERS:\n        value = getattr(args, name)\n        if value is not None:\n            result[name] = value\n    for setting in args.param:\n        name, separator, raw = setting.partition(\'=\')\n        if not separator:\n            raise ValueError(\'--param must have the form KEY=JSON\')\n        result[name] = json.loads(raw)\n    return validate_parameters(result)\n', encoding='utf-8')
(OCR_WORK / 'unlimited_ocr_app.py').write_text('"""Drive-native Unlimited-OCR 2 job server (Caddy protects the loopback API)."""\nfrom __future__ import annotations\n\nimport hashlib\nimport base64\nimport json\nimport os\nimport queue\nimport re\nimport tempfile\nimport threading\nimport time\nimport uuid\nimport shutil\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\nfrom ocr_parameters import validate_parameters, inference_kwargs\n\nEXTENSIONS = {\'.pdf\', \'.png\', \'.jpg\', \'.jpeg\', \'.webp\', \'.bmp\', \'.tif\', \'.tiff\'}\nDRIVE = Path(os.environ.get(\'CT_DRIVE_ROOT\', \'/content/drive/MyDrive\')).resolve()\nDATA = Path(os.environ.get(\'CT_DATA_DIR\', str(DRIVE / \'colab-llm/unlimited-ocr\')))\nMODEL_REVISION = \'0480966938291445591f69f08a13b041c073a96a\'\n\n\ndef drive_path(spec: str) -> Path:\n    if not isinstance(spec, str) or not spec.strip():\n        raise ValueError(\'The Drive path cannot be empty\')\n    spec = spec.removeprefix(\'drive://\')\n    if spec == \'MyDrive\':\n        spec = \'.\'\n    elif spec.startswith(\'MyDrive/\'):\n        spec = spec[len(\'MyDrive/\'):]\n    path = Path(spec)\n    path = (path if path.is_absolute() else DRIVE / path).resolve()\n    if not path.is_relative_to(DRIVE):\n        raise ValueError(\'The path must be inside /content/drive/MyDrive\')\n    return path\n\n\ndef atomic_json(path: Path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    try:\n        temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n        os.replace(temporary, path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef discover(specs, recursive=True, output=None):\n    if not isinstance(specs, list) or not specs:\n        raise ValueError(\'Specify at least one Drive file or folder\')\n    found = set()\n    for spec in specs:\n        root = drive_path(spec)\n        if not root.exists():\n            raise ValueError(f\'Input does not exist: {spec}\')\n        candidates = (root.rglob(\'*\') if recursive else root.iterdir()) if root.is_dir() else [root]\n        for candidate in candidates:\n            path = candidate.resolve()\n            if not path.is_relative_to(DRIVE):\n                continue\n            if output and path.is_relative_to(output):\n                continue\n            if path.is_file() and path.suffix.lower() in EXTENSIONS:\n                found.add(path)\n    if not found:\n        raise ValueError(\'No PDF or supported image found\')\n    return sorted(found)\n\n\ndef options(body):\n    dpi = int(body.get(\'dpi\', 200))\n    pages = int(body.get(\'pages_per_chunk\', 10))\n    tokens = int(body.get(\'tokens_per_page\', 8000))\n    if not 72 <= dpi <= 400 or not 1 <= pages <= 100 or not 32 <= tokens <= 20000:\n        raise ValueError(\'dpi: 72–400; pages_per_chunk: 1–100; tokens_per_page: 32–20000\')\n    return dict(dpi=dpi, pages_per_chunk=pages, tokens_per_page=tokens,\n                inference_params=validate_parameters(body.get(\'inference_params\', {})))\n\n\ndef file_signature(source, opts):\n    stat = source.stat()\n    return hashlib.sha256(json.dumps([str(source), stat.st_size, stat.st_mtime_ns,\n                                     opts, MODEL_REVISION]).encode()).hexdigest()\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with path.open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\nclass Checkpoints:\n    """Commit a page group only after raw text, Markdown and all assets are durable."""\n    def __init__(self, destination, signature):\n        self.destination = Path(destination)\n        self.root = self.destination / \'.checkpoints\'\n        self.signature = signature\n        try:\n            current = json.loads((self.root / \'state.json\').read_text())\n        except (OSError, ValueError):\n            current = {}\n        if current.get(\'signature\') != signature:\n            shutil.rmtree(self.root, ignore_errors=True)\n            # These names are exclusively generated by this service.\n            for path in self.destination.iterdir():\n                if re.fullmatch(r\'(chunk|native)-[0-9]+\', path.name):\n                    if path.is_symlink():\n                        path.unlink()\n                    elif path.is_dir():\n                        shutil.rmtree(path)\n            atomic_json(self.root / \'state.json\', dict(signature=signature, version=1))\n\n    def load(self, start, end):\n        try:\n            record = json.loads((self.root / f\'{start:06d}.json\').read_text())\n            if (record[\'signature\'] != self.signature or record[\'end\'] != end\n                    or not isinstance(record[\'raw\'], str) or not isinstance(record[\'markdown\'], str)\n                    or type(record[\'tokens\']) is not int or record[\'tokens\'] < 0):\n                return None\n            for name, digest in record[\'artifacts\'].items():\n                path = (self.destination / name).resolve()\n                if not path.is_relative_to(self.destination.resolve()) or sha256_file(path) != digest:\n                    return None\n            return record\n        except (OSError, ValueError, KeyError, TypeError, AttributeError):\n            return None\n\n    def save(self, start, end, raw, markdown, tokens, artifacts):\n        record = dict(signature=self.signature, end=end, raw=raw, markdown=markdown, tokens=tokens,\n                      artifacts={str(p.relative_to(self.destination)): sha256_file(p) for p in artifacts})\n        atomic_json(self.root / f\'{start:06d}.json\', record)\n        return record\n\n\nclass Engine:\n    def __init__(self):\n        import torch\n        from transformers import AutoModel, AutoTokenizer\n        self.torch = torch\n        model_dir = os.environ[\'CT_OCR_MODEL_DIR\']\n        self.tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)\n        self.model = AutoModel.from_pretrained(\n            model_dir, trust_remote_code=True, use_safetensors=True,\n            torch_dtype=torch.bfloat16).eval().cuda()\n        # Compile and exercise the actual GPU path before health can report ready.\n        from PIL import Image, ImageDraw\n        image = Image.new(\'RGB\', (512, 512), \'white\')\n        ImageDraw.Draw(image).text((32, 32), \'OCR ready\', fill=\'black\')\n        with torch.inference_mode():\n            self.model.infer(self.tokenizer, image_file=image, eval_mode=True,\n                             max_new_tokens=32, crop_mode=False)\n\n    def infer_pages(self, pages, native_output, opts):\n        # infer_multi forces return_dict=True and rewrites prompts; infer preserves all options.\n        params = inference_kwargs(opts.get(\'inference_params\', {}), len(pages), opts[\'tokens_per_page\'])\n        with self.torch.inference_mode():\n            result = self.model.infer(self.tokenizer, image_file=[str(p) for p in pages],\n                                      output_path=str(native_output), **params)\n        if isinstance(result, dict):\n            return result[\'text\'], len(result[\'token_ids\'])\n        return result, len(self.tokenizer.encode(result, add_special_tokens=False))\n\n    def convert(self, source: Path, destination: Path, opts, progress):\n        import fitz\n        from PIL import Image\n        from process_unlimitedOCR2 import convert\n        destination.mkdir(parents=True, exist_ok=True)\n        raw_chunks, markdown_chunks = [], []\n        total_tokens = 0\n        resumed_chunks = 0\n        # Overwrite uses a per-job generation: restart resumes that overwrite, a new job starts fresh.\n        checkpoint_signature = hashlib.sha256(json.dumps([\n            file_signature(source, {k: v for k, v in opts.items() if not k.startswith(\'_\')}),\n            opts.get(\'_checkpoint_generation\', \'shared\'), 1]).encode()).hexdigest()\n        checkpoints = Checkpoints(destination, checkpoint_signature)\n        with tempfile.TemporaryDirectory(prefix=\'ocr-\', dir=\'/content/unlimited-ocr\') as tmp:\n            # Render only the current page window, keeping RAM and local disk bounded.\n            pdf = fitz.open(source) if source.suffix.lower() == \'.pdf\' else None\n            raster = Image.open(source) if pdf is None else None\n            try:\n                page_count = len(pdf) if pdf is not None else getattr(raster, \'n_frames\', 1)\n                for start in range(0, page_count, opts[\'pages_per_chunk\']):\n                    end = min(start + opts[\'pages_per_chunk\'], page_count)\n                    saved = checkpoints.load(start, end)\n                    if saved is not None:\n                        raw_chunks.append(saved[\'raw\'])\n                        markdown_chunks.append(saved[\'markdown\'])\n                        total_tokens += saved[\'tokens\']\n                        resumed_chunks += 1\n                        progress(end, page_count)\n                        continue\n                    # A group without a valid commit may contain interrupted/stale assets.\n                    for prefix in (\'chunk\', \'native\'):\n                        shutil.rmtree(destination / f\'{prefix}-{start + 1:04d}\', ignore_errors=True)\n                    pages = []\n                    for index in range(start, min(start + opts[\'pages_per_chunk\'], page_count)):\n                        path = Path(tmp) / f\'page{index + 1}.png\'\n                        if pdf is not None:\n                            pdf[index].get_pixmap(dpi=opts[\'dpi\'], alpha=False).save(path)\n                        else:\n                            raster.seek(index)\n                            raster.convert(\'RGB\').save(path)\n                        pages.append(path)\n                    raw, count = self.infer_pages(pages, destination / f\'native-{start + 1:04d}\', opts)\n                    total_tokens += count\n                    # Each chunk has its own asset directory; upstream Markdown page indices restart at 1.\n                    assets = destination / f\'chunk-{start + 1:04d}\'\n                    assets.mkdir(exist_ok=True)\n                    markdown = convert(raw)\n                    for name, page, coords in re.findall(\n                            r\'(page(\\d+)_((?:\\d+_){3}\\d+)\\.jpg)\', markdown):\n                        idx = int(page) - 1\n                        if 0 <= idx < len(pages):\n                            with Image.open(pages[idx]) as image:\n                                x1, y1, x2, y2 = map(int, coords.split(\'_\'))\n                                if x2 > x1 and y2 > y1:\n                                    image.crop((x1 * image.width / 1000, y1 * image.height / 1000,\n                                                x2 * image.width / 1000, y2 * image.height / 1000)).convert(\'RGB\').save(assets / name)\n                    markdown = re.sub(r\'!\\[\\]\\((page\\d+_[\\d_]+\\.jpg)\\)\',\n                                      lambda m: f\'![]({assets.name}/{m[1]})\', markdown)\n                    artifacts = [p for prefix in (\'chunk\', \'native\')\n                                 for p in (destination / f\'{prefix}-{start + 1:04d}\').rglob(\'*\') if p.is_file()]\n                    checkpoints.save(start, end, raw, markdown, count, artifacts)\n                    raw_chunks.append(raw)\n                    markdown_chunks.append(markdown)\n                    for path in pages:\n                        path.unlink()\n                    progress(min(start + len(pages), page_count), page_count)\n                if not page_count:\n                    raise ValueError(\'The document has no pages\')\n            finally:\n                if pdf is not None:\n                    pdf.close()\n                if raster is not None:\n                    raster.close()\n        (destination / \'result.raw.txt\').write_text(\'\\n\'.join(raw_chunks), encoding=\'utf-8\')\n        (destination / \'result.md\').write_text(\'\\n\\n\'.join(markdown_chunks), encoding=\'utf-8\')\n        return dict(pages=page_count, tokens=total_tokens, chunks=len(raw_chunks), resumed_chunks=resumed_chunks)\n\n\nclass Jobs:\n    def __init__(self, engine, data=DATA, start_worker=True):\n        self.engine = engine\n        self.root = Path(data) / \'jobs\'\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.lock = threading.Lock()\n        self.queue = queue.Queue()\n        recovered = []\n        for path in self.root.glob(\'*.json\'):\n            try:\n                record = json.loads(path.read_text())\n                if record[\'status\'] in (\'queued\', \'running\'):\n                    self.prepare_resume(record, automatic=True)\n                    self.save(record)\n                    recovered.append(record)\n            except (OSError, ValueError, KeyError, TypeError) as error:\n                print(f\'[recovery] skipped invalid job record {path.name}: {error}\', flush=True)\n        for record in sorted(recovered, key=lambda item: item.get(\'created\', 0)):\n            self.queue.put(record)\n        if start_worker:\n            threading.Thread(target=self.worker, daemon=True).start()\n\n    @staticmethod\n    def prepare_resume(record, automatic=False):\n        # Validate persisted settings before putting them back on the GPU queue.\n        record[\'options\'] = options(record[\'options\'])\n        if not record.get(\'files\') or not re.fullmatch(r\'[0-9a-f]{32}\', record[\'id\']):\n            raise ValueError(\'Invalid job record\')\n        record.update(status=\'queued\', resume_count=record.get(\'resume_count\', 0) + 1,\n                      resumed_at=time.time(), automatic_resume=automatic)\n        record.pop(\'error\', None)\n        record.pop(\'finished\', None)\n        for item in record[\'files\']:\n            if item[\'status\'] not in (\'completed\', \'skipped\'):\n                item[\'status\'] = \'queued\'\n                item.pop(\'error\', None)\n\n    def resume(self, job_id):\n        # One lock covers validation and persisted transition, so duplicate resume cannot enqueue twice.\n        if not re.fullmatch(r\'[0-9a-f]{32}\', job_id):\n            raise ValueError(\'Invalid job ID\')\n        with self.lock:\n            path = self.root / (job_id + \'.json\')\n            record = json.loads(path.read_text())\n            if record[\'status\'] in (\'queued\', \'running\', \'completed\'):\n                raise ValueError(\'The job is already queued, running or completed\')\n            self.prepare_resume(record)\n            atomic_json(path, record)\n            self.queue.put(record)\n        return record\n\n    def get(self, job_id):\n        if not re.fullmatch(r\'[0-9a-f]{32}\', job_id):\n            raise ValueError(\'Invalid job ID\')\n        with self.lock:\n            return json.loads((self.root / (job_id + \'.json\')).read_text())\n\n    def save(self, record):\n        with self.lock:\n            atomic_json(self.root / (record[\'id\'] + \'.json\'), record)\n\n    def submit(self, body):\n        opts = options(body)\n        output = drive_path(body.get(\'output_dir\') or str(DATA / \'outputs\'))\n        paths = discover(body.get(\'inputs\'), body.get(\'recursive\', True), output)\n        if any(output == path or output.is_relative_to(path) for path in paths):\n            raise ValueError(\'The output directory cannot be inside an input file\')\n        job = dict(id=uuid.uuid4().hex, status=\'queued\', created=time.time(), options=opts,\n                   output_dir=str(output), overwrite=bool(body.get(\'overwrite\', False)),\n                   files=[dict(source=str(p), status=\'queued\',\n                               relative_output=body.get(\'relative_outputs\', {}).get(str(p), str(p.relative_to(DRIVE)))) for p in paths])\n        for item in job[\'files\']:\n            relative = Path(item[\'relative_output\'])\n            if relative.is_absolute() or \'..\' in relative.parts or not relative.parts:\n                raise ValueError(\'Invalid relative output path\')\n        if len({item[\'relative_output\'] for item in job[\'files\']}) != len(paths):\n            raise ValueError(\'Relative output paths of several source files conflict\')\n        self.save(job)\n        self.queue.put(job)\n        return job\n\n    def worker(self):\n        while True:\n            job = self.queue.get()\n            try:\n                self.run(job)\n            except Exception as error:\n                job.update(status=\'failed\', error=str(error))\n                self.save(job)\n            finally:\n                self.queue.task_done()\n\n    def run(self, job):\n        job[\'status\'] = \'running\'\n        self.save(job)\n        for item in job[\'files\']:\n            source = Path(item[\'source\'])\n            destination = drive_path(str(Path(job[\'output_dir\']) / item.get(\'relative_output\', str(source.relative_to(DRIVE)))))\n            marker = destination / \'complete.json\'\n            try:\n                signature = file_signature(source, job[\'options\'])\n                if (not job[\'overwrite\'] or item.get(\'completed_signature\') == signature) and marker.is_file():\n                    previous = json.loads(marker.read_text())\n                    if previous.get(\'signature\') == signature and (destination / \'result.md\').is_file():\n                        item.update(status=\'skipped\', output=str(destination))\n                        self.save(job)\n                        continue\n                # An old completion marker must never validate a partially overwritten result.\n                marker.unlink(missing_ok=True)\n                item[\'status\'] = \'running\'\n                item.pop(\'error\', None)\n                self.save(job)\n                def progress(done, total):\n                    item.update(pages_done=done, pages_total=total)\n                    self.save(job)\n                conversion_opts = {**job[\'options\'],\n                                   \'_checkpoint_generation\': job[\'id\'] if job[\'overwrite\'] else \'shared\'}\n                result = self.engine.convert(source, destination, conversion_opts, progress)\n                if file_signature(source, job[\'options\']) != signature:\n                    raise ValueError(\'A source file changed during conversion; resume the job to reprocess it\')\n                atomic_json(marker, dict(signature=signature, source=str(source), options=job[\'options\'], **result))\n                item.update(status=\'completed\', output=str(destination), completed_signature=signature, **result)\n            except Exception as error:\n                item.update(status=\'failed\', error=str(error))\n            self.save(job)\n        job.update(status=\'failed\' if any(f[\'status\'] == \'failed\' for f in job[\'files\']) else \'completed\',\n                   finished=time.time())\n        self.save(job)\n\n\ndef upload_chunk(body):\n    """Stage local input in a content-addressed Drive inbox, in bounded chunks."""\n    digest = body.get(\'sha256\', \'\')\n    name = body.get(\'name\', \'\')\n    if not re.fullmatch(r\'[0-9a-f]{64}\', digest) or Path(name).name != name or Path(name).suffix.lower() not in EXTENSIONS:\n        raise ValueError(\'Invalid upload name or SHA256\')\n    offset = int(body.get(\'offset\', 0))\n    chunk = base64.b64decode(body.get(\'data\', \'\'), validate=True)\n    if offset < 0 or len(chunk) > 512 * 1024:\n        raise ValueError(\'Invalid chunk size/offset\')\n    folder = drive_path(str(DATA / \'uploads\' / digest))\n    folder.mkdir(parents=True, exist_ok=True)\n    destination = folder / name\n    temporary = folder / (name + \'.part\')\n    if destination.exists():\n        return dict(path=str(destination), complete=True)\n    if offset == 0:\n        temporary.write_bytes(chunk)\n    else:\n        if not temporary.exists() or temporary.stat().st_size != offset:\n            raise ValueError(\'Upload offset mismatch; retry the upload\')\n        with temporary.open(\'ab\') as stream:\n            stream.write(chunk)\n    if body.get(\'final\'):\n        hasher = hashlib.sha256()\n        with temporary.open(\'rb\') as stream:\n            for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n                hasher.update(block)\n        if hasher.hexdigest() != digest:\n            temporary.unlink()\n            raise ValueError(\'Upload SHA256 verification failed\')\n        os.replace(temporary, destination)\n    return dict(path=str(destination), complete=destination.exists())\n\n\ndef result_files(jobs, job_id, index, name=None, offset=0):\n    record = jobs.get(job_id)\n    item = record[\'files\'][index]\n    if item[\'status\'] not in (\'completed\', \'skipped\'):\n        raise ValueError(\'The file has not finished converting\')\n    root = drive_path(item[\'output\'])\n    if name is None:\n        return dict(files=[str(p.relative_to(root)) for p in sorted(root.rglob(\'*\'))\n                           if p.is_file() and \'.checkpoints\' not in p.relative_to(root).parts\n                           and p.resolve().is_relative_to(root)])\n    target = (root / name).resolve()\n    if not target.is_relative_to(root) or not target.is_file() or offset < 0:\n        raise ValueError(\'Invalid result file path/offset\')\n    with target.open(\'rb\') as stream:\n        stream.seek(offset)\n        chunk = stream.read(512 * 1024)\n    return dict(data=base64.b64encode(chunk).decode(), next_offset=offset + len(chunk),\n                complete=offset + len(chunk) >= target.stat().st_size)\n\n\ndef serve():\n    engine = Engine()\n    jobs = Jobs(engine)\n\n    class Handler(BaseHTTPRequestHandler):\n        def send_json(self, code, value):\n            raw = json.dumps(value, ensure_ascii=False).encode()\n            self.send_response(code)\n            self.send_header(\'Content-Type\', \'application/json; charset=utf-8\')\n            self.send_header(\'Content-Length\', str(len(raw)))\n            self.end_headers()\n            self.wfile.write(raw)\n\n        def do_GET(self):\n            try:\n                if self.path in (\'/\', \'/health\'):\n                    return self.send_json(200, dict(status=\'ready\', model=\'baidu/Unlimited-OCR-2\'))\n                if self.path.startswith(\'/api/jobs/\'):\n                    return self.send_json(200, jobs.get(self.path.removeprefix(\'/api/jobs/\')))\n                self.send_json(404, dict(error=\'not found\'))\n            except FileNotFoundError:\n                self.send_json(404, dict(error=\'job not found\'))\n            except ValueError as error:\n                self.send_json(400, dict(error=str(error)))\n\n        def do_POST(self):\n            try:\n                size = int(self.headers.get(\'Content-Length\', \'0\'))\n                if not 0 < size <= 1024 * 1024:\n                    raise ValueError(\'Invalid request length\')\n                body = json.loads(self.rfile.read(size))\n                if not isinstance(body, dict):\n                    raise ValueError(\'JSON body must be an object\')\n                if self.path == \'/api/upload\':\n                    return self.send_json(200, upload_chunk(body))\n                if self.path == \'/api/results\':\n                    return self.send_json(200, result_files(jobs, body[\'job_id\'], int(body[\'index\']),\n                        body.get(\'name\'), int(body.get(\'offset\', 0))))\n                if re.fullmatch(r\'/api/jobs/[0-9a-f]{32}/resume\', self.path):\n                    return self.send_json(202, jobs.resume(self.path.split(\'/\')[3]))\n                if self.path == \'/api/jobs\':\n                    return self.send_json(202, jobs.submit(body))\n                if self.path == \'/api/drive-list\':\n                    return self.send_json(200, dict(files=[str(p.relative_to(DRIVE)) for p in\n                        discover(body.get(\'inputs\'), body.get(\'recursive\', True))]))\n                self.send_json(404, dict(error=\'not found\'))\n            except (ValueError, TypeError, OSError, KeyError, IndexError) as error:\n                self.send_json(400, dict(error=str(error)))\n\n    ThreadingHTTPServer((\'127.0.0.1\', int(os.environ.get(\'CT_SERVICE_PORT\', \'8000\'))), Handler).serve_forever()\n\n\nif __name__ == \'__main__\':\n    serve()\n', encoding='utf-8')
def start_service():
    return ct_spawn("service", [OCR_PY, str(OCR_WORK / 'unlimited_ocr_app.py')], cwd=str(OCR_WORK),
                    env={"CT_SERVICE_PORT": str(SERVICE_PORT), "CT_DATA_DIR": DATA_DIR,
                         "CT_OCR_MODEL_DIR": str(OCR_MODEL),
                         "TRITON_CACHE_DIR": str(OCR_WORK / 'triton-cache'),
                         "TORCHINDUCTOR_CACHE_DIR": str(OCR_WORK / 'inductor-cache'),
                         "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
PROCS["service"] = (start_service(), start_service)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)
import platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- KEEP THIS CELL RUNNING
# The Colab session only lives while the `colab exec` connection is open and this cell blocks.
# The watchdog restarts any process from PROCS that died; a heartbeat line is printed every 10 min.
ct_wait_http(f"http://127.0.0.1:{SERVICE_PORT}/health", timeout=5400)
if AUTH == "caddy" and not CREDS_FROM_ENV:   # started from the Colab UI: show generated credentials
    print(f"CT_CREDENTIALS user={AUTH_USER} pass={AUTH_PASS}")
    print(f"API token: {API_TOKEN}")
for local in OCR_CACHE_PAIRS:
    try:
        save_cache(local, Path(DATA_DIR) / 'kernel-archives', OCR_CACHE_KEY)
    except (OSError, ValueError) as err:
        print(f'Cache archive save warning ({local.name}): {err}', flush=True)
print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

last_beat = time.time()
while True:
    time.sleep(30)
    for name, (proc, restart) in list(PROCS.items()):
        if proc.poll() is not None:
            print(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 15)}", flush=True)
            ct_stop_proc(proc)
            PROCS[name] = (restart(), restart)
    if time.time() - last_beat > 600:
        last_beat = time.time()
        print(f"[heartbeat] {time.strftime('%H:%M:%S')} " +
              " ".join(f"{n}={'up' if p.poll() is None else 'down'}" for n, (p, _) in PROCS.items()), flush=True)
